In [ ]:
import pandas as pd 
import numpy as np 
from olist_analysis import load, config, cleaning
from pathlib import Path

เรื่องการหาค่า CI(confidence interval) ของคะแนนรีวิว

In [ ]:
cfg = config.ReportConfig

review = load.read_table('order_reviews', cfg.raw_dir)
r =review['review_score']

n = r.count()
mean = r.mean()
se = r.std(ddof=1)/np.sqrt(n)
low, high = mean - 1.96*se, mean + 1.96*se 

print(f'n = {n:,} mean = {mean:.3f} se = {se:.3f} CI = {low:.3f} - {high:.3f}')

In [ ]:
customer = load.read_table('customer', cfg.raw_dir)
order = load.read_table('orders', cfg.raw_dir)
clean = cleaning.clean_orders(order)


ประชากรกับกลุ่มตัวอย่าง

ร้านกาแฟมีคนหนึ่งมาซื้อทุกวัน 30 วัน กับอีก 30 คนที่มาครั้งเดียว ถ้านับแบบ “ต่อการซื้อ” คนประจำมีเสียง 30 เสียง เท่ากับอีก 30 คนรวมกัน
ถ้านับแบบ “ต่อคน” ทุกคนมีหนึ่งเสียงเท่ากัน

สองวิธีนี้ให้คำตอบต่างกันได้ ต้องเลือกให้ตรงกับคำถาม สิ่งที่ใช้นับนี้เรียกว่า หน่วยวิเคราะห์ (unit of analysis)

In [ ]:
print(len(customer))
print(customer['customer_id'].nunique())
print(customer['customer_unique_id'].nunique())

การสุ่มในแบบต่างๆ 

In [ ]:
pop = order.merge(customer[['customer_id', 'customer_state']],
                on = 'customer_id', how = 'left')

k = 300

sample = {
    'ก. หยิบจากบนลงมา': pop.sort_values('order_purchase_timestamp').head(k),
    'ข. หยิบแบบสุ่ม' : pop.sample(n=k, random_state=42),
    'ค. หยิบแบบสุ่มตามสัดส่วน': pop.groupby('customer_state').sample(frac=k/len(pop), random_state=42),
    'ประชากร' : pop 
}

for name, s in sample.items():
    print(f"{name:24s} n={len(s):>6,} รัฐที่เจอ={s['customer_state'].nunique():>2}"
        f"สัดส่วน SP={(s['customer_state'] == 'SP').mean():.1%}")

3.4 Sampling bias: ใครเงียบ

ภาพ: ร้านแปะกล่องรับความคิดเห็นไว้หน้าร้าน เดือนนี้มีใบติชมสิบใบ เขียนว่าอาหารช้าทั้งหมด ร้านควรสรุปไหมว่าลูกค้าส่วนใหญ่ไม่พอใจเรื่องความเร็ว

ลูกค้าเป็นพันคนที่กินเสร็จแล้วเดินออกไปเฉย ๆ ไม่ได้อยู่ในกล่องนี้ ข้อมูลที่ได้มาจากคนที่เลือกเขียนเอง ปัญหานี้เรียกว่า non-response bias (คนที่ไม่ตอบ ต่างจากคนที่ตอบ)

วิธีตรวจ: แยกคนที่เขียน กับคนที่เงียบ แล้วดูว่าต่างกันไหม ในเรื่องที่เรารู้ข้อมูลของทั้งสองกลุ่ม

In [ ]:
rev = review[['order_id', 'review_score', 'review_comment_message']]
d = clean[clean['delivery_days'].notna()].merge(rev, on='order_id', how="inner")

d['has_comment'] = d['review_comment_message'].notna()
d['is_late'] = d['days_late'] > 0

d.groupby('has_comment').agg(
    n=('order_id', 'size'),
    late_rate=('is_late', 'mean'),
    mean_score=('review_score','mean')
)